In [0]:
loans_repay_raw_df = spark.read \
.format("csv") \
.option("header", True) \
.option("inferSchema",True) \
.load("/Volumes/finance_domain_project/default/raw_data/Raw/loans_repayments_csv")


In [0]:
loans_repay_schema = "loan_id string, total_principal_received float,total_interest_received float,total_late_fee_received float, total_payment_received float,last_payment_amount float,last_payment_date string,next_payment_date string"

In [0]:
loans_repay_raw_df = spark.read \
.format("csv") \
.option("header", True) \
.schema(loans_repay_schema) \
.load("/Volumes/finance_domain_project/default/raw_data/Raw/loans_repayments_csv")


In [0]:
from pyspark.sql.functions import current_timestamp


In [0]:
loans_repay_df_ingested = loans_repay_raw_df.withColumn("ingest_date",current_timestamp())

In [0]:
loans_repay_df_ingested.createOrReplaceTempView("loan_repayments")

In [0]:
display(spark.sql("select * from loan_repayments"))

In [0]:
display(spark.sql("select count(*) from loan_repayments where total_principal_received is null"))

In [0]:
columns_to_check=["total_principal_received","total_interest_received","total_late_fee_received","total_payment_received","last_payment_amount"]


In [0]:
loans_repay_filtered_df = loans_repay_df_ingested.na.drop(subset=columns_to_check)


In [0]:
loans_repay_filtered_df.count()

In [0]:
loans_repay_filtered_df.createOrReplaceTempView("loan_repayments")

In [0]:
display(spark.sql("select * from loan_repayments where total_payment_received = 0.0"))

In [0]:
display(spark.sql("select count(*) from loan_repayments where total_payment_received = 0.0"))

In [0]:
display(spark.sql("select count(*) from loan_repayments where total_payment_received = 0.0 and total_principal_received !=0.0"))

In [0]:
from pyspark.sql.functions import when,col

In [0]:
loans_payment_fixed_df = loans_repay_filtered_df.withColumn("total_payment_received", when(
    (col("total_principal_received")!=0.0)&
    (col("total_payment_received")==0.0), col("total_principal_received")+ col("total_interest_received")+col("total_late_fee_received")
).otherwise(col("total_payment_received"))
)


In [0]:
loans_payment_fixed2_df = loans_payment_fixed_df.filter("total_payment_received != 0.0") 


In [0]:
loans_payment_fixed2_df.filter(col("last_payment_date") == "0.0").count()


In [0]:
loans_payment_fixed2_df.filter(col("next_payment_date") == "0.0").count()


In [0]:
from pyspark.sql.functions import col, when, trim


In [0]:
loans_payment_last_date_fixed_df = loans_payment_fixed2_df.withColumn(
    "last_payment_date",
    when(trim(col("last_payment_date")).isin("0", "0.0"), None)
    .otherwise(col("last_payment_date"))
)

In [0]:
loans_payment_next_date_fixed_df = loans_payment_last_date_fixed_df.withColumn(
    "next_payment_date",
    when(trim(col("next_payment_date")).isin("0", "0.0"), None)
    .otherwise(col("next_payment_date"))
)

In [0]:
loans_payment_next_date_fixed_df.filter(
    trim(col("next_payment_date")).isin("0", "0.0")
).count()

In [0]:
loans_payment_next_date_fixed_df.write \
.option("header",True) \
.format("csv") \
.mode("overwrite") \
.option("path","/Volumes/finance_domain_project/default/raw_data/Cleaned/loans_repayments_csv") \
.save()


In [0]:
loans_payment_next_date_fixed_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path","/Volumes/finance_domain_project/default/raw_data/Cleaned/loans_repayments_parquet") \
.save()
